# Example: Using the Digital Twin Policy Learning Interface

This notebook illustrates how to use the generic policy learning interface in
`digital_twin_policy_learning.py` with the included facsimile dataset.

The workflow includes:

1. loading the example data
2. preparing the trajectory dataset
3. initializing the learner
4. training or loading the sequence model
5. training tabular Q-learning
6. evaluating example policies

In [ ]:
import pandas as pd
import numpy as np

from digital_twin_policy_learning import (
    TrajectoryDataset,
    MicrosimQLearner
)

## Load facsimile data
This dataset should follow the same structure as the paper’s EHR data
(id, time, action, covariates, outcomes, etc.)

In [ ]:
# Replace with your facsimile dataset (same structure as paper data)
df = pd.read_csv("your_facsimile_data.csv")

df.head()

## Define column roles

In [ ]:
patient_id_col = "id"
time_col = "month_index"
action_col = "action"

# RNN inputs
rnn_covariate_cols = [
    # your covariates (must include action_col)
    "action",
    "Age",
    "Gender",
    "Visits",
    "imm_baseline",
    # ...
]

# RNN outputs
rnn_outcome_cols = [
    "sev_inf_next",
    "inf_next"
]

# RL state (discretized)
rl_state_cols = [
    "age_cat",
    "imm_baseline",
    "months_since_vax_cat"
]

## Build dataset

In [ ]:
dataset = TrajectoryDataset.from_long_format(
    df=df,
    patient_id_col=patient_id_col,
    time_col=time_col,
    action_col=action_col,
    rnn_covariate_cols=rnn_covariate_cols,
    rnn_outcome_cols=rnn_outcome_cols,
    rl_state_cols=rl_state_cols,

    # optional (user-defined)
    reward_outcome_col="sev_inf_next"
)

dataset.summary()

## Define optional custom functions

In [ ]:
# ===== USER-DEFINED COMPONENTS =====

def reward_fn(context):
    """
    your function:
    define reward based on predicted outcomes
    
    Example (paper-style):
    reward = - severe_risk * (1 + vax_cost * action)
    """
    pass


def action_constraint_fn(context):
    """
    your function:
    enforce constraints (e.g., no vaccine within 4 months)
    """
    pass


def terminal_fn(context):
    """
    your function:
    define terminal condition (e.g., severe infection)
    """
    pass

## Initialize learner

In [ ]:
learner = MicrosimQLearner(
    dataset=dataset,

    # optional hooks
    reward_fn=None,            # replace with reward_fn if needed
    action_constraint_fn=None,
    terminal_fn=None
)

## Train OR load RNN

In [ ]:
learner.fit_sequence_model(
    hidden_size=128,
    num_layers=2,
    epochs=2000,
    lr=1e-4
)

In [ ]:
learner.load_sequence_model(
    model_path="your_rnn_weights.pth"
)

## Train Q-learning

In [ ]:
q_result = learner.fit_tabular_q_learning(
    repeats_train_eval=30,
    gamma=0.99
)

## Evaluate policies

In [ ]:
learned = learner.evaluate_policy("learned", epochs=5)
observed = learner.evaluate_policy("observed", epochs=5)
always = learner.evaluate_policy("all", epochs=5)
never = learner.evaluate_policy("none", epochs=5)

## Compare results

In [ ]:
print("learned:", np.mean(learned))
print("observed:", np.mean(observed))
print("always:", np.mean(always))
print("never:", np.mean(never))